# Day 19 — 후보 유전자를 한 개씩 들여다보기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-3/day-19.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-3/day-19.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 평균은 같아도 이야기는 다르다

In [ ]:
import numpy as np
import pandas as pd

gc = np.array([8.0, 8.1, 7.9, 8.0, 8.1, 7.9])
flt_a = np.array([7.2, 7.3, 7.1, 7.2, 7.3, 7.1])     # 여섯 마리 모두 낮음
flt_b = np.array([8.0, 8.1, 7.9, 8.0, 8.1, 3.1])     # 한 마리만 크게 낮음

for name, flt in [("geneA", flt_a), ("geneB", flt_b)]:
    full = flt.mean() - gc.mean()
    without_min = np.delete(flt, flt.argmin()).mean() - gc.mean()
    print(f"{name}: log2FC = {full:.2f} | 가장 낮은 한 마리를 빼면 {without_min:.2f}")

### 예시 2 — 후보가 패널에 있는지 집합으로 확인하기

In [ ]:
panel = {"Tnf", "Il1b", "Trem2", "Mog", "Plp1", "Casp3"}      # 설명용 작은 패널
wish_list = ["Trem2", "Gfap", "Mog", "TNF"]

print("패널에 있음:", [g for g in wish_list if g in panel])
print("패널에 없음:", [g for g in wish_list if g not in panel])

### 예시 3 — 점을 흩뜨려 찍는 가장 작은 그림

In [ ]:
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(3, 3))
for x, values, color in [(0, gc, "tab:blue"), (1, flt_b, "tab:red")]:
    jitter = rng.uniform(-0.08, 0.08, size=len(values))
    ax.scatter(x + jitter, values, color=color)
    ax.hlines(values.mean(), x - 0.2, x + 0.2, color="black")      # 군 평균
ax.set_xticks([0, 1], ["GC", "FLT"])
ax.set_ylabel("log2 expression")
ax.set_title("geneB (toy data)")
fig.tight_layout()
print("찍은 점의 수:", len(gc) + len(flt_b))

## 직접 해보기

1. 설정 셀과 저장해 둔 표를 불러옵니다.

In [ ]:
# 이 셀은 Day 15–18의 산출물 파일이 필요합니다. Colab은 노트북마다 런타임이 따로라서,
# 그 파일을 이 런타임에 업로드하거나 Google Drive에 두고 경로를 맞추세요.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MODE = "practice"          # 연습이 끝나면 "real"로 바꾸세요
RAW_DIR = Path("../data/osdr_536_practice" if MODE == "practice" else "../data/osdr_536")
OUT_DIR = RAW_DIR / "processed"
FIG_DIR = Path("../figures")

log2_norm = pd.read_csv(OUT_DIR / "log2_norm.csv", index_col=0)
gene_qc = pd.read_csv(OUT_DIR / "gene_qc.csv", index_col=0)
de = pd.read_csv(OUT_DIR / "de_table.csv").set_index("gene")
samples = pd.read_csv(OUT_DIR / "samples.csv").set_index("sample_id").loc[log2_norm.index]
print(log2_norm.shape, de.shape)

2. 범주별 후보를 정합니다. 연습 모드의 범주는 가짜 이름표입니다. 실제 모드의 목록은 **출발점**입니다. 교과서적 기능을 근거로 고른 a priori 후보이고, 모두 패널에 존재함을 2026-11-12에 RCC 파일에서 확인했습니다. 여러분의 판단으로 바꾸고, 어제의 `top_candidates.csv`에서 관심 가는 유전자를 `data_driven`에 추가해 전체가 5–10개가 되게 하세요.

In [ ]:
if MODE == "practice":
    candidates = {
        "inflammation (fake)": ["Prac01", "Prac02"],
        "oxidative stress (fake)": ["Prac11", "Prac12"],
        "synapse (fake)": ["Prac21", "Prac22"],
        "myelin (fake)": ["Prac31", "Prac32"],
        "apoptosis (fake)": ["Prac41", "Prac42"],
        "data_driven": [],
    }
else:
    candidates = {
        "neuroinflammation": ["Trem2", "C3"],
        "oxidative stress": ["Nfe2l2", "Hmox1"],
        "synapse": ["Syp", "Dlg4"],
        "myelin": ["Mog", "Plp1"],
        "apoptosis": ["Casp3", "Bax"],
        "data_driven": [],          # 여기에 어제 표에서 고른 유전자 기호를 넣으세요
    }

cand = pd.DataFrame([(g, c) for c, genes in candidates.items() for g in genes],
                    columns=["gene", "category"])
cand["basis"] = np.where(cand["category"] == "data_driven", "data-driven", "a priori")
print(len(cand), "개 후보")

3. 후보마다 패널 존재 여부, 검출 안정성, 어제의 통계값을 붙여 한 표로 만듭니다.

In [ ]:
cand["in_panel"] = cand["gene"].isin(log2_norm.columns)
cand["detected"] = cand["gene"].map(gene_qc["keep"]).fillna(False).astype(bool)
cand = cand.join(de[["log2FC", "p_value", "FDR"]], on="gene")
print(cand.round(3).to_string(index=False))
cand.round(4).to_csv(OUT_DIR / "candidate_genes.csv", index=False)

4. 후보 유전자마다 작은 그림 하나씩, 샘플 하나가 점 하나가 되게 그립니다.

In [ ]:
plot_genes = cand[cand["in_panel"]].reset_index(drop=True)
n_cols = 5
n_rows = int(np.ceil(len(plot_genes) / n_cols))
rng = np.random.default_rng(0)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.6 * n_cols, 3 * n_rows), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")

for ax, row in zip(axes.ravel(), plot_genes.itertuples()):
    ax.axis("on")
    for x, g, color in [(0, "GC", "tab:blue"), (1, "FLT", "tab:red")]:
        values = log2_norm.loc[samples["group"] == g, row.gene]
        ax.scatter(x + rng.uniform(-0.08, 0.08, len(values)), values, color=color, s=25)
        ax.hlines(values.mean(), x - 0.25, x + 0.25, color="black", linewidth=1)
    ax.set_xticks([0, 1], ["GC", "FLT"])
    ax.set_xlim(-0.6, 1.6)
    stats_text = f"log2FC {row.log2FC:.2f}, FDR {row.FDR:.2f}" if row.detected else "low detection"
    ax.set_title(f"{row.gene}\n{row.category}\n{stats_text}", fontsize=8)

for ax in axes[:, 0]:
    ax.set_ylabel("log2 normalized count")
fig.suptitle(f"Candidate genes, one dot per animal (exploratory) [data: {MODE}]", fontsize=10)
fig.tight_layout()
fig.savefig(FIG_DIR / f"osdr_{MODE}_gene_level.png", dpi=150)
print("그린 유전자 수:", len(plot_genes))

5. 그림에서 점 하나가 어느 동물인지 알고 싶을 때는 값을 표로 꺼냅니다. 아래는 첫 번째 후보의 예입니다.

In [ ]:
gene = plot_genes.loc[0, "gene"]
view = samples[["source", "group"]].assign(log2=log2_norm[gene].round(2))
print(gene)
print(view.sort_values(["group", "log2"]).to_string(index=False))